In [1]:
!pip install torch_scatter torcheeg torch_geometric -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 kB 13.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.4/233.4 kB 19.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 86.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 61.7 MB/s 

In [2]:
import torch
import torch.nn as nn
import torch.nn.utils as utils
from torch.utils.data import DataLoader, Subset ,WeightedRandomSampler
from torcheeg.models import CCNN
from torcheeg import transforms
from torcheeg.transforms import ToGrid
from torcheeg.datasets import SEEDIVDataset,SEEDIVFeatureDataset
from torcheeg.datasets.constants import SEED_IV_CHANNEL_LOCATION_DICT
from torcheeg.transforms import ToG
from torcheeg.datasets.constants import SEED_IV_ADJACENCY_MATRIX
from torcheeg.models import DGCNN
import torch_geometric.loader as geom_loader # Special loader for graphs
import copy
import scipy.signal as signal
import random
import numpy as np
import os

In [5]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [6]:
dataset = SEEDIVFeatureDataset(
    io_path='./tmp_out/seed_iv_features',
    root_path='/kaggle/input/seed-iv/eeg_feature_smooth',
    feature=['de_LDS'], 
    num_worker=0,
    offline_transform=transforms.Compose([
        transforms.To2d(), 
        transforms.Lambda(lambda x: torch.tensor(x).float())]),
    label_transform=transforms.Select('emotion'),
)

[2026-06-27 10:04:12] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_features.
[2026-06-27 10:04:12] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]:   0%|          | 0/45 [00:00<?, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 0it [00:00, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 1it [00:00,  9.52it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 24it [00:00, 135.00it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 49it [00:00, 184.24it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 74it [00:00, 208.87it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 99it [00:00, 221.47it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_201511

In [17]:
def map_emotions(y):
    if y == 1 or y == 2 :  # Sad or Fear
        return 0 
    if y == 3:  # Happy
        return 1  
    return -1 # Neutral

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    """Basic Graph Convolution Layer."""
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj):
        # x: (B, N, Fin), adj: (B, N, N)
        support = torch.matmul(x, self.weight)
        output = torch.matmul(adj, support)
        return F.relu(output)

class MultiGraphicLayerConstruction(nn.Module):
    """Transformer-based adaptive adjacency matrix construction."""
    def __init__(self, feature_dim, num_heads=2, k=5):
        super(MultiGraphicLayerConstruction, self).__init__()
        self.k = k
        self.num_heads = num_heads
        self.norm1 = nn.LayerNorm(feature_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=num_heads, batch_first=True)
        
        self.norm2 = nn.LayerNorm(feature_dim)
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim, feature_dim * 2),
            nn.ReLU(),
            nn.Linear(feature_dim * 2, feature_dim)
        )

       # داخل كلاس MultiGraphicLayerConstruction
    def forward(self, x):
        B, N, D = x.shape
        x_norm = self.norm1(x)
        
        # 1. Scaled Dot-Product
        scaling = D ** 0.5
        _, attn_weights = self.multihead_attn(x_norm, x_norm, x_norm)
        
        attn_out, _ = self.multihead_attn(x_norm, x_norm, x_norm)
        x2 = self.norm2(x + attn_out)
        g_feature = self.mlp(x2) + x2
        
        # 2. Scaling الـ Matrix multiplication قبل الـ Softmax
        s_matrix = torch.matmul(g_feature, g_feature.transpose(-1, -2)) / scaling
        
        # 3. Adjacency Matrix
        adj = F.softmax(attn_weights + s_matrix, dim=-1)
        
        # 4. Top-K Sparsity (مهم جداً للـ Stability)
        # تأكد إن الـ K مش كبيرة أوي (10 مناسبة جداً لـ 62 قناة)
        topk_val, _ = torch.topk(adj, self.k, dim=-1)
        mask = (adj >= topk_val[:, :, -1].unsqueeze(-1)).float()
        adj = adj * mask
        
        return adj, g_feature

class BFENet_SingleBand(nn.Module):
    """Implementation of BFE-Net for one frequency band."""
    def __init__(self, num_channels=62):
        super(BFENet_SingleBand, self).__init__()
        
        # CNN Layer: Processes single-channel features independently
        # Input (B, 1, 1) -> Paper implies processing the DE feature dimension
        # Since input is (B, 62, 5), for a single band it's (B, 62, 1)
        self.cnn1 = nn.Conv1d(1, 64, kernel_size=1) 
        self.cnn2 = nn.Conv1d(64, 128, kernel_size=1)
        self.cnn3 = nn.Conv1d(128, 256, kernel_size=1)
        
        self.graph_const = MultiGraphicLayerConstruction(feature_dim=256)
        self.gcn = GCNLayer(256, 128)
        self.dropout = nn.Dropout(0)

    def forward(self, x):
        # x: (B, 62, 1) -> treat 62 as nodes, 1 as feature
        B, N, _ = x.shape
        x = x.transpose(1, 2) # (B, 1, 62)
        
        # CNN Feature Extraction
        x = F.relu(self.cnn1(x))
        x = F.relu(self.cnn2(x))
        x = F.relu(self.cnn3(x)) # (B, 256, 62)
        
        x = x.transpose(1, 2) # (B, 62, 256) -> (B, Nodes, Features)
        
        # Adaptive Graph Construction
        adj, g_features = self.graph_const(x)
        
        # Graph Convolution
        x_gcn = self.gcn(g_features, adj) # (B, 62, 128)
        
        # Flatten for fusion
        x_flat = x_gcn.reshape(B, -1) 
        return x_flat

class BFENet_Full(nn.Module):
    """The full model that fuses 5 frequency bands."""
    def __init__(self, num_classes=3): # 3 for SEED, 4 for SEED-IV
        super(BFENet_Full, self).__init__()
        self.bands = nn.ModuleList([BFENet_SingleBand() for _ in range(5)])
        
        # Final classification layer
        # Input: 5 bands * (62 nodes * 128 gcn features)
        self.classifier = nn.Sequential(
            nn.Linear(5 * 62 * 128, 256),
            nn.ReLU(),
            nn.Dropout(0),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x shape: (B, 62, 5)
        band_features = []
        for i in range(5):
            band_data = x[:, :, i].unsqueeze(-1) # (B, 62, 1)
            band_out = self.bands[i](band_data)
            band_features.append(band_out)
        
        # Fusion
        merged = torch.cat(band_features, dim=1)
        logits = self.classifier(merged)
        return logits



In [21]:
# Parameters
EPOCHS = 200
BATCH_SIZE = 64
LR = 5e-5
PATIENCE_LR = 3
REDUCE_FACTOR = 0.5
PATIENCE_ES = 20
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

In [22]:
import torch
import torch.nn as nn
import numpy as np
import os
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

meta_info = dataset.info
subjects = sorted(meta_info['subject_id'].unique())

if not os.path.exists("saved_models"):
    os.makedirs("saved_models")

results = {}

print("===== Subject Independent Evaluation (2-Class) =====")

for test_subject in subjects:

    print(f"\n{'='*40}")
    print(f" Testing on Subject {test_subject}")
    print(f"{'='*40}")

    # ================= SPLIT DATA =================
    train_df = meta_info[meta_info['subject_id'] != test_subject].copy()
    test_df  = meta_info[meta_info['subject_id'] == test_subject].copy()

    # Map emotions
    train_df['emotion_mapped'] = train_df['emotion'].apply(map_emotions)
    test_df['emotion_mapped']  = test_df['emotion'].apply(map_emotions)

    # Remove Neutral
    train_df = train_df[train_df['emotion_mapped'] != -1]
    test_df  = test_df[test_df['emotion_mapped'] != -1]

    train_indices = train_df.index.tolist()
    test_indices  = test_df.index.tolist()

    print(f"Train Samples: {len(train_indices)} | Test Samples: {len(test_indices)}")

    train_set = Subset(dataset, train_indices)
    test_set  = Subset(dataset, test_indices)

    # ================= SAMPLER =================
    y_train = train_df['emotion_mapped'].values
    class_counts = np.bincount(y_train)
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = [class_weights[y] for y in y_train]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=0,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_set,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    # ================= MODEL =================
    model = BFENet_Full(num_classes=2).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_acc = 0.0
    patience_counter = 0

    # ================= TRAINING =================
    for epoch in range(EPOCHS):
        model.train()
        train_correct = 0
        train_total = 0
        train_loss = 0
        batch_count = 0

        for X, y in train_loader:
            X = X.to(device)

            # Map labels again (safe)
            y_bin = torch.tensor([map_emotions(v.item()) for v in y], device=device)

            # Remove neutral if exists
            mask = y_bin != -1
            X = X[mask]
            y_bin = y_bin[mask]

            if len(X) == 0:
                continue

            if len(X.shape) == 4:
                X = X.squeeze(1)   # (B,62,5)

            # Normalize per sample
            mean = X.mean(dim=(1,2), keepdim=True)
            std  = X.std(dim=(1,2), keepdim=True) + 1e-6
            X = (X - mean) / std

            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y_bin.long())
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            batch_count += 1
            _, preds = torch.max(outputs, 1)
            train_total += y_bin.size(0)
            train_correct += (preds == y_bin).sum().item()

        train_acc = 100 * train_correct / train_total
        train_loss /= batch_count

        # ================= TEST =================
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0
        batch_count = 0

        with torch.no_grad():
            for X, y in test_loader:
                X = X.to(device)
                y_bin = torch.tensor([map_emotions(v.item()) for v in y], device=device)

                mask = y_bin != -1
                X = X[mask]
                y_bin = y_bin[mask]

                if len(X) == 0:
                    continue

                if len(X.shape) == 4:
                    X = X.squeeze(1)

                mean = X.mean(dim=(1,2), keepdim=True)
                std  = X.std(dim=(1,2), keepdim=True) + 1e-6
                X = (X - mean) / std

                outputs = model(X)
                loss = criterion(outputs, y_bin.long())

                val_loss += loss.item()
                batch_count += 1
                _, preds = torch.max(outputs, 1)
                val_total += y_bin.size(0)
                val_correct += (preds == y_bin).sum().item()

        val_acc = 100 * val_correct / val_total
        val_loss /= batch_count
        scheduler.step()

        print(f"Epoch {epoch+1:03d} | Train Acc {train_acc:.2f}% | Val Acc {val_acc:.2f}%")

        # ================= EARLY STOPPING =================
        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), f"saved_models/subject_{test_subject}_best.pth")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE_ES:
                print("Early stopping.")
                break

    print(f"Subject {test_subject} Best Accuracy = {best_acc:.2f}%")
    results[test_subject] = best_acc


# ================= FINAL REPORT =================
print("\n" + "="*50)
print(" FINAL SUBJECT-INDEPENDENT RESULTS (2-Class) ")
print("="*50)

all_accs = list(results.values())

for sid in results:
    print(f"Subject {sid:02d}: {results[sid]:.2f}%")

print(f"\nAverage Accuracy = {np.mean(all_accs):.2f}%")
best_subject = max(results, key=results.get)
print(f"Best Subject: {best_subject} ({results[best_subject]:.2f}%)")
print("="*50)

===== Subject Independent Evaluation (2-Class) =====

 Testing on Subject 1
Train Samples: 25578 | Test Samples: 1827
Epoch 001 | Train Acc 76.51% | Val Acc 74.22%
Epoch 002 | Train Acc 95.64% | Val Acc 71.59%
Epoch 003 | Train Acc 99.34% | Val Acc 75.26%
Epoch 004 | Train Acc 99.71% | Val Acc 79.42%
Epoch 005 | Train Acc 99.90% | Val Acc 73.73%
Epoch 006 | Train Acc 99.98% | Val Acc 74.88%
Epoch 007 | Train Acc 99.97% | Val Acc 73.84%
Epoch 008 | Train Acc 100.00% | Val Acc 74.77%
Epoch 009 | Train Acc 100.00% | Val Acc 74.06%
Epoch 010 | Train Acc 100.00% | Val Acc 74.71%
Epoch 011 | Train Acc 100.00% | Val Acc 74.77%
Epoch 012 | Train Acc 99.70% | Val Acc 78.11%
Epoch 013 | Train Acc 100.00% | Val Acc 78.76%
Epoch 014 | Train Acc 100.00% | Val Acc 76.74%
Epoch 015 | Train Acc 100.00% | Val Acc 78.43%
Epoch 016 | Train Acc 100.00% | Val Acc 79.47%
Epoch 017 | Train Acc 99.91% | Val Acc 77.61%
Epoch 018 | Train Acc 100.00% | Val Acc 79.31%
Epoch 019 | Train Acc 100.00% | Val Acc 78.54

In [23]:
print("\nFinal Subject-Independent Results")
for s in results:
    print(f"Subject {s}: {results[s]:.2f}%")

print(f"\nAverage Accuracy: {np.mean(list(results.values())):.2f}%")


Final Subject-Independent Results
Subject 1: 79.47%
Subject 2: 51.18%
Subject 3: 68.14%
Subject 4: 78.98%
Subject 5: 75.64%
Subject 6: 66.89%
Subject 7: 81.17%
Subject 8: 80.13%
Subject 9: 76.03%
Subject 10: 73.84%
Subject 11: 80.46%
Subject 12: 82.05%
Subject 13: 41.11%
Subject 14: 83.31%
Subject 15: 82.81%

Average Accuracy: 73.41%
